# 05 · 하이퍼파라미터 튜닝 (팀원용)

**돌리기 전에 [`docs/HPO_GUIDE.md`](https://github.com/hyunku9566/lga_data/blob/main/docs/HPO_GUIDE.md) 를 읽어라.**

---

## 30초 요약

- **모델 설정 변경은 v7 이후 7전 7패다.** 성공한 건 피처 추가(`pbc_*` LB +3.30) 하나뿐이다.
  HPO 는 "큰 이득" 이 아니라 "현행이 여전히 최적인지 확인" 으로 다뤄라.
- **기준선은 항상 현재 제출본(v16wB)** 이다. 피처 124개(`build_v16`), XGB d10/mcw6000/n2000/lr.005.
  낡은 기준선(v10a, 순정 baseline 586.97)에 대고 재면 그 수치는 전이되지 않는다.
- **시드 3개 이상.** 노이즈 막대 없는 델타는 보고하지 않는다.
- 두 폴드 동시 개선은 **필요조건이지 충분조건이 아니다** (v17 이 둘 다 양수였는데 LB -5.84).

## 이미 끝난 것 — 다시 하지 마라

| 대상 | 결과 |
|---|---|
| XGB HPO | 2회 완료(23/27차). 현행이 최적 |
| LGB / CatBoost 재튜닝 | CV 좋았으나 **LB -5.84** (v17) |
| 성분모델(cmp_*) 재튜닝 | **기각** — 성분 예측은 좋아졌는데 y 가 나빠짐 (`results29/`) |
| 블렌드 가중치 CV 최적화 | **금지** — CV +14.2 -> LB -11.66 |
| 드리프트 스칼라 | 최적 -0.0206 확정 |

## 아직 안 건드린 것 — 여기부터

1. **NN 축 재튜닝** (트리가 810 이던 시절 설정 그대로). 단 천장이 +2 다.
2. **시드 수 / 앙상블 다양성** — 분산 축소라 부호가 뒤집힐 여지가 적다.
3. **최근성 반감기 `hl`** — 현행 2.0. 폴드2024 에서 hl=1.0 이 +5.6 로 더 좋았는데
   ABS 전제를 근거로 2.0 을 택했고, **그 전제는 반증됐다**(`docs/CEILING.md` 4절).


In [ ]:
# ── 부트스트랩 · 이 셀을 가장 먼저 실행 ──────────────────────────
# 코랩은 노트북마다 런타임(VM)이 다르다. 01 에서 받아둔 것은 여기 없다.
# 이 셀 하나가 레포 클론 → 의존성 → 데이터 확보까지 전부 처리한다.
#   데이터는 로컬 → Drive → 데이터 서버 순으로 찾고, 서버에서 받은 건 Drive 에 백업한다.
RUNNER_NAME = "본인이름"        # ← 여기만 바꾼다

import os, sys, subprocess
if not os.path.exists('/content/lga-repo/src/config.py'):
    subprocess.run(['git','clone','-q',
                    'https://github.com/hyunku9566/lga_data.git','/content/lga-repo'])
else:
    subprocess.run(['git','-C','/content/lga-repo','pull','-q'])
subprocess.run(['pip','install','-q','-r','/content/lga-repo/requirements-colab.txt'])

for m in [k for k in list(sys.modules)
          if k.startswith('src') or k in ('config','lib_lga','experiment','download','bootstrap')]:
    del sys.modules[m]
for _p in ('/content/lga-repo/src', '/content/lga-repo'):
    if _p not in sys.path: sys.path.insert(0, _p)

# 서버에서 받아야 할 수도 있으니 비밀번호를 미리 받아둔다 (Drive 에 있으면 안 쓴다)
if not os.environ.get('LGA_PASSWORD'):
    from getpass import getpass
    os.environ['LGA_PASSWORD'] = getpass('team 비밀번호 (Drive 에 캐시 있으면 엔터): ').strip()

from bootstrap import setup
C = setup(runner=RUNNER_NAME)
import lib_lga as L, experiment as E


In [ ]:
if 'E' not in globals(): exec(open('/content/lga-repo/colab/_ensure.py').read())

SEEDS = 3                       # 최소 3. 2면 노이즈 막대가 커서 대부분 '노이즈' 판정이 난다

base = E.get_baseline()         # 기준선 = v16 124피처 + 현행 XGB, 시드 5
print(f"기준선  폴드2024 {base['m24']:.1f} ±{base['m24_sd']:.1f}   폴드2023 {base['m23']:.1f} ±{base['m23_sd']:.1f}")
print(f"피처 {base['nfeat']}개")
print(f"시드 {SEEDS} 기준 노이즈 막대(2se): 2024 ±{2*E._se(base['m24_sd'],SEEDS):.1f}  2023 ±{2*E._se(base['m23_sd'],SEEDS):.1f}")
print('\n이 막대보다 작은 차이는 전부 노이즈다. 보고하지 마라.')

---
## 스윕 1 · XGB 현행 재확인

23/27차에서 이미 튜닝된 지점이다. **이득을 기대하지 말고, 현행이 여전히 최적인지만 확인한다.**
현행 밖으로 크게 벗어난 조합이 이기면 그때 의미가 있다.

조합당 3~6분 · 아래 격자는 9조합 ≈ 40분

In [ ]:
r = E.run_experiment(
    name   = 'xgb_현행재확인',
    kind   = 'hparam',
    grid   = {'max_depth': [8, 10, 12], 'min_child_weight': [3000, 6000, 12000]},
    model  = 'xgb',
    seeds  = SEEDS,
    runner = RUNNER_NAME,
    notes  = '현행 d10/mcw6000 이 여전히 최적인지 확인',
)
r

---
## 스윕 2 · 최근성 반감기 `hl` — **우선순위 높음**

현행 `hl=2.0` 은 "2024 ABS regime" 전제를 근거로 골랐는데 **그 전제가 반증됐다**
(R리그에 2024 단절이 없다 — `docs/CEILING.md` 4절). 폴드2024 단독으로는 `hl=1.0` 이 +5.6 였다.

`hl` 은 격자가 아니라 폴드 컨텍스트 인자라 `bench2` 로 직접 돈다.

In [ ]:
import lib_lga as L, numpy as np, pandas as pd, time
X = L.build_v16()
rows = []
for hl in (1.0, 1.5, 2.0, 3.0):
    out = {}
    for vs in (2024, 2023):
        ctx = L.fold_ctx(vs, hl=hl)
        ps = [L.fit_predict(X, L.load_base()['y'], E.DEFAULT_XGB, ctx, nseed=1) for _ in range(1)]
        out[vs] = L.bss(np.mean(ps, 0), ctx['yv'], ctx['base'])
    rows.append(dict(hl=hl, m24=out[2024], m23=out[2023]))
    print(f"  hl={hl:<4} 2024 {out[2024]:8.1f}   2023 {out[2023]:8.1f}")
pd.DataFrame(rows)

---
## 스윕 3 · LGB / CatBoost

**주의: 49~56차에서 찾은 신설정은 CV 가 크게 좋았는데 LB 에서 -5.84 였다(v17).**
같은 지점을 다시 파는 것은 의미가 없다. 돌린다면 **전혀 다른 영역**을 봐라.

`ref_params` 를 안 넘기면 해당 부스터의 현행 설정이 자동으로 자기 기준선이 된다.

In [ ]:
r = E.run_experiment(
    name   = 'lgb_미탐색영역',
    kind   = 'hparam',
    grid   = {'num_leaves': [7, 15, 31], 'reg_lambda': [10., 50., 200.]},
    model  = 'lgb',
    seeds  = SEEDS,
    runner = RUNNER_NAME,
    notes  = 'v17 에서 실패한 extra_trees/cs0.4 영역은 제외',
)
r

---
## 결과 커밋

원장은 `ledgers/<본인이름>.csv` 에 쌓인다. 아래 셀이 커밋·푸시한다.
전원 결과를 모아 보려면 `04_summary.ipynb`.

In [ ]:
import sync
sync.push(runner=RUNNER_NAME)

---
## 채택후보가 나왔다면

**바로 제출하지 마라.** `bench2` 는 XGB 단독을 재는데 제출본은 4축 블렌드다.
희석률이 균일하지 않다 — `pc_*` 는 폴드2023 에서 0.15배로 소멸했고 `pbc_*` 는 0.65배로 살아남았다.
그 차이가 두 후보의 LB 성패를 갈랐다 (`pc_*` -1.52 vs `pbc_*` +3.30).

**블렌드 수준에서 다시 재고 나서 제출한다.** 팀 채널에 결과를 올리고 상의해라.